# 📺 Sony Bravia KLV-24P413D Cloud Media Converter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OddBoyXD/bravia-tv-converter/blob/main/Sony_Bravia_TV_Converter_Colab.ipynb)

Transcode any movie or direct link (GoFile, Pixeldrain, Direct URL) specifically for the **Sony Bravia KLV-24P413D USB media player**.

### ⚡ Features Included:
1. **⏱️ Live Visual Progress & ETA**: Real-time progress bar with time remaining, current encoding speed, and elapsed time.
2. **🚀 Automatic GPU Detection (Feature 5)**: Automatically detects and enables **NVIDIA T4 GPU NVENC** for ultra-fast ~8 to 10 minute encoding, or seamlessly falls back to multi-threaded CPU mode.
3. **☕ Colab Keep-Alive (Feature 6)**: Background anti-disconnect script keeps the session active so the tab won't sleep.
4. **🎵 Smart Multi-Audio Track Selector**: Automatically detects all audio streams and pre-selects the **Hindi track** (`hin`, `hi`, `Hindi`, or clean audio).
5. **🛡️ Strict Resolution & Quality Invariant**: Preserves native resolution (never upscales 480p/576p to 720p, cleanly downscales 1080p+ to 720p).
6. **📦 Automatic GitHub Release**: Automatically uploads the finished MP4 directly to [`OddBoyXD/bravia-tv-converter`](https://github.com/OddBoyXD/bravia-tv-converter) Releases (<2 GB safe cap).

---

In [ ]:
# @title ⚡ Step 1: Keep-Alive & GPU Auto-Detection (Feature 5 & 6)
# @markdown Run this cell first. It detects GPU acceleration and keeps the Colab tab active.

import os
import subprocess
import sys
from IPython.display import display, HTML, clear_output

# Feature 6: Colab Keep-Alive Script (Prevents tab disconnect/sleep)
display(HTML('''
<script>
function KeepAlive() {
    console.log("☕ Colab Keep-Alive ping: " + new Date().toLocaleTimeString());
    document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect")?.click();
}
setInterval(KeepAlive, 60000);
console.log("☕ Colab Keep-Alive active! Tab will stay awake.");
</script>
'''))

# Feature 5: Automatic GPU Hardware Detection
has_gpu = False
gpu_name = "None"
try:
    res = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode == 0 and res.stdout.strip():
        has_gpu = True
        gpu_name = res.stdout.strip().split('\n')[0]
except Exception:
    pass

print("==================================================")
if has_gpu:
    print(f"🚀 GPU ACCELERATION ACTIVE: {gpu_name}")
    print("⚡ Encoder: NVIDIA NVENC Hardware Transcoder (Super Fast ~8-10 min)")
    ENCODER_TYPE = "h264_nvenc"
else:
    print("💻 CPU MODE ACTIVE: Standard Cloud CPU")
    print("ℹ️ Encoder: libx264 veryfast (Multi-threaded CPU)")
    print("💡 TIP: For 5x faster speed, switch to GPU in Runtime -> Change runtime type -> T4 GPU")
    ENCODER_TYPE = "libx264"
print("==================================================")

# Verify Python dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'tqdm', 'ipywidgets'], check=False)
print("✅ Setup ready! Proceed to Step 2.")


In [ ]:
# @title 🔍 Step 2: Link Resolver & Multi-Audio Track Selector
# @markdown Paste your GoFile link or direct media URL below.

import os
import re
import json
import time
import hashlib
import requests
import subprocess
import ipywidgets as widgets
from IPython.display import display

MEDIA_URL = "https://gofile.io/d/..." #@param {type:"string"}
CUSTOM_TITLE = "" #@param {type:"string"}

GOFILE_USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
GOFILE_LANGUAGE = "en-US"
GOFILE_WT_SALT = "12af056dacea0b"
WT_WINDOW_SECONDS = 14400

def generate_gofile_token(account_token: str, window_offset: int = 0) -> str:
    window = int(time.time() // WT_WINDOW_SECONDS) + window_offset
    raw = f"{GOFILE_USER_AGENT}::{GOFILE_LANGUAGE}::{account_token}::{window}::{GOFILE_WT_SALT}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def extract_gofile_id(url: str) -> str:
    m = re.search(r'gofile\.io/d/([a-zA-Z0-9_-]+)', url)
    if m: return m.group(1)
    m = re.search(r'gofile\.io/\?c=([a-zA-Z0-9_-]+)', url)
    if m: return m.group(1)
    return url.strip().split('?')[0].rstrip('/').split('/')[-1]

def collect_gofile_files(node):
    files = []
    if not isinstance(node, dict): return files
    if (node.get("type") == "file" or node.get("link")) and node.get("link"):
        files.append(node)
    children = node.get("children") or node.get("contents") or {}
    if isinstance(children, dict):
        for child in children.values(): files.extend(collect_gofile_files(child))
    elif isinstance(children, list):
        for child in children: files.extend(collect_gofile_files(child))
    return files

def resolve_gofile(url: str):
    content_id = extract_gofile_id(url)
    print(f"[*] Resolving GoFile content ID: {content_id}...")
    session = requests.Session()
    headers = {'User-Agent': GOFILE_USER_AGENT, 'Referer': 'https://gofile.io/'}
    try: session.get(f"https://gofile.io/d/{content_id}", headers=headers, timeout=15)
    except: pass
    
    account_token = None
    try:
        r = session.post("https://api.gofile.io/accounts", headers={'User-Agent': GOFILE_USER_AGENT}, timeout=10)
        acc = r.json()
        if acc.get("status") == "ok": account_token = acc.get("data", {}).get("token")
    except: pass
    
    for offset in [0, -1, 1]:
        wt = generate_gofile_token(account_token or "", window_offset=offset)
        api_headers = {
            'User-Agent': GOFILE_USER_AGENT,
            'Accept': 'application/json',
            'X-BL': GOFILE_LANGUAGE,
            'X-Website-Token': wt,
            'Referer': f"https://gofile.io/d/{content_id}"
        }
        if account_token: api_headers['Authorization'] = f"Bearer {account_token}"
        try:
            resp = session.get(f"https://api.gofile.io/contents/{content_id}", headers=api_headers, params={'wt': wt, 'cache': 'true'}, cookies=session.cookies.get_dict(), timeout=15)
            data = resp.json()
            if data.get("status") == "ok":
                files = collect_gofile_files(data.get("data", {}))
                if files:
                    files.sort(key=lambda x: x.get("size", 0), reverse=True)
                    chosen = files[0]
                    return chosen.get("link"), chosen.get("name", "video.mp4"), session, account_token
        except: pass
    raise RuntimeError("Failed to resolve GoFile link. Verify URL.")

direct_url = MEDIA_URL.strip()
session_cookies = None
account_token = None
raw_filename = "media_input.mp4"

if "gofile.io" in direct_url:
    direct_url, raw_filename, session, account_token = resolve_gofile(direct_url)
    session_cookies = session.cookies.get_dict()
else:
    raw_filename = direct_url.split('?')[0].rstrip('/').split('/')[-1] or "media_input.mp4"

print(f"[*] Direct Media URL: {direct_url[:80]}...")
print(f"[*] Media File Name: {raw_filename}")

# Fast Container Probe
probe_sample = "sample_probe.bin"
headers = {'User-Agent': GOFILE_USER_AGENT, 'Range': 'bytes=0-26214400'}
if account_token: headers['Authorization'] = f"Bearer {account_token}"

print("[*] Probing video & audio stream metadata...")
r = requests.get(direct_url, headers=headers, cookies=session_cookies, stream=True, timeout=25)
with open(probe_sample, 'wb') as f:
    for chunk in r.iter_content(chunk_size=65536):
        f.write(chunk)

ffprobe_cmd = ['ffprobe', '-v', 'error', '-show_streams', '-show_format', '-of', 'json', probe_sample]
res = subprocess.run(ffprobe_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
meta = json.loads(res.stdout) if res.returncode == 0 else {}

video_streams = [s for s in meta.get("streams", []) if s.get("codec_type") == "video"]
audio_streams = [s for s in meta.get("streams", []) if s.get("codec_type") == "audio"]

in_width = int(video_streams[0].get("width", 1280)) if video_streams else 1280
in_height = int(video_streams[0].get("height", 720)) if video_streams else 720
print(f"\n📺 Source Resolution: {in_width}x{in_height}")

if in_width <= 1280 and in_height <= 720:
    print(f"🛡️ Resolution Invariant: Keeping native {in_width}x{in_height} (ZERO fake upscaling ✅)")
else:
    print(f"🛡️ Resolution Invariant: Downscaling cleanly from {in_width}x{in_height} to 720p native panel limit ✅")

print("\n🎵 Detected Audio Tracks in File:")
track_options = []
default_selected_idx = 0

for i, a in enumerate(audio_streams):
    lang = (a.get("tags", {}).get("language", "") or a.get("tags", {}).get("lang", "") or "und").lower()
    title = a.get("tags", {}).get("title", "")
    codec = a.get("codec_name", "aac")
    ch = a.get("channels", 2)
    bitrate = a.get("bit_rate")
    br_str = f" @ {int(bitrate)//1000}k" if bitrate else ""
    
    label = f"[Track {i}] {lang.upper()}: {title or 'Audio'} ({codec}, {ch}ch{br_str})"
    track_options.append((label, i))
    
    # Auto-detect Hindi (hin, hi, hindi, or Hindi title)
    if lang in ["hin", "hi", "hindi"] or "hindi" in title.lower() or "hin" in title.lower():
        default_selected_idx = i

if not track_options:
    track_options = [("[Track 0] Default Audio", 0)]

print(f"👉 Auto-Selected Track: Track {default_selected_idx} (Hindi priority)")

audio_selector = widgets.RadioButtons(
    options=track_options,
    value=default_selected_idx,
    description='Audio Track:',
    disabled=False,
    layout={'width': 'max-content'}
)
display(audio_selector)


In [ ]:
# @title ⚡ Step 3: Transcode for Sony Bravia (Live Progress & ETA - Feature 1)
# @markdown Runs FFmpeg hardware-constrained conversion with real-time ETA progress.

import os
import re
import time
import requests
import subprocess
from tqdm.auto import tqdm

SELECTED_AUDIO_INDEX = audio_selector.value
print(f"[*] Active Audio Track Index: {SELECTED_AUDIO_INDEX}")

base_title = CUSTOM_TITLE.strip() or os.path.splitext(raw_filename)[0] or "Sony_Bravia_Video"
clean_title = re.sub(r'[^a-zA-Z0-9_\.-]', '_', base_title)[:50]
final_output = f"{clean_title}_SonyBravia_720p.mp4"
raw_download = "input_full.bin"

# Download full source if needed
if not os.path.exists(raw_download) or os.path.getsize(raw_download) < 1000000:
    print(f"[*] Downloading full source file: {raw_filename}...")
    dl_headers = {'User-Agent': GOFILE_USER_AGENT}
    if account_token: dl_headers['Authorization'] = f"Bearer {account_token}"
    
    resp = requests.get(direct_url, headers=dl_headers, cookies=session_cookies, stream=True, timeout=60)
    total_bytes = int(resp.headers.get('content-length', 0))
    
    with open(raw_download, 'wb') as f, tqdm(
        desc="📥 Downloading Media",
        total=total_bytes,
        unit='B',
        unit_scale=True,
        unit_divisor=1024
    ) as bar:
        for chunk in resp.iter_content(chunk_size=1048576):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))
    print("✅ Download complete!")

# Probe total video duration for progress tracking
p_cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', raw_download]
p_res = subprocess.run(p_cmd, stdout=subprocess.PIPE, text=True)
try:
    total_duration = float(p_res.stdout.strip())
except:
    total_duration = 3600.0

total_mins = int(total_duration // 60)
total_secs = int(total_duration % 60)
print(f"⏱️ Video Duration: {total_mins:02d}m {total_secs:02d}s")

# Long Movie Safety Bitrate (Keeps under 2.0 GiB GitHub release limit)
crf_val = "22"
if total_duration > 9000: # Over 2.5 hours
    crf_val = "23"
    print("🛡️ Extended movie detected: Automatically tuned CRF to 23 (guarantees < 2.0 GB upload cap ✅)")

# Assemble FFmpeg command matching exact Sony Bravia KLV-24P413D profile
ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-i", raw_download,
    "-map", "0:v:0",
    "-map", f"0:a:{SELECTED_AUDIO_INDEX}",
    "-vf", "scale='min(1280,iw)':-2:flags=lanczos",
    "-r", "30"
]

if ENCODER_TYPE == "h264_nvenc":
    ffmpeg_cmd.extend([
        "-c:v", "h264_nvenc",
        "-profile:v", "main",
        "-level", "3.1",
        "-pix_fmt", "yuv420p",
        "-preset", "p4",
        "-cq", crf_val
    ])
else:
    ffmpeg_cmd.extend([
        "-c:v", "libx264",
        "-profile:v", "main",
        "-level", "3.1",
        "-pix_fmt", "yuv420p",
        "-preset", "veryfast",
        "-crf", crf_val
    ])

ffmpeg_cmd.extend([
    "-c:a", "aac",
    "-b:a", "192k",
    "-ac", "2",
    "-ar", "44100",
    "-movflags", "+faststart",
    "-progress", "pipe:1",
    final_output
])

print("\n🚀 Starting Sony Bravia Hardware Transcoding...")
start_time = time.time()
process = subprocess.Popen(ffmpeg_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Feature 1: Live Progress Bar with ETA and Speed
with tqdm(
    total=int(total_duration),
    desc="⚡ Transcoding (Sony Bravia 720p)",
    unit="s",
    bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt}s [{elapsed}<{remaining}, {rate_fmt}]"
) as pbar:
    prev_secs = 0
    for line in process.stdout:
        line = line.strip()
        if line.startswith("out_time_ms="):
            try:
                cur_us = int(line.split("=")[1])
                cur_secs = int(cur_us / 1000000)
                delta = cur_secs - prev_secs
                if delta > 0:
                    pbar.update(min(delta, int(total_duration) - pbar.n))
                    prev_secs = cur_secs
            except: pass
        elif line.startswith("progress=end"):
            pbar.update(int(total_duration) - pbar.n)
            break

process.wait()
if process.returncode != 0:
    raise RuntimeError("FFmpeg transcoding failed.")

out_bytes = os.path.getsize(final_output)
out_mb = out_bytes / (1024 * 1024)
out_gb = out_bytes / (1024 * 1024 * 1024)
elapsed = time.time() - start_time

print(f"\n🎉 CONVERSION COMPLETED in {elapsed:.1f}s!")
print(f"📦 Output File: {final_output}")
print(f"📊 Final File Size: {out_mb:.2f} MB ({out_gb:.2f} GB)")
if out_bytes < 2147483648:
    print("✅ Validated: Output is strictly under 2.0 GiB (100% safe for GitHub Releases)!")


In [ ]:
# @title 📦 Step 4: Publish to GitHub Releases (OddBoyXD/bravia-tv-converter)
# @markdown Publishes the final TV-ready MP4 as a single release and gives you a 1-click download link.

from datetime import datetime
from google.colab import userdata
import requests

REPO = "OddBoyXD/bravia-tv-converter"

# Fetch token from Colab Secrets or prompt
gh_token = None
try:
    gh_token = userdata.get('GITHUB_TOKEN')
except:
    pass

if not gh_token:
    from getpass import getpass
    gh_token = getpass("Paste your GitHub Personal Access Token: ").strip()

if not gh_token:
    raise ValueError("GitHub Token required to create a release.")

headers = {
    "Authorization": f"Bearer {gh_token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

tag_name = f"tv-build-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
rel_title = f"Sony Bravia TV: {clean_title} ({tag_name})"

print(f"[*] Creating release '{tag_name}' on {REPO}...")
payload = {
    "tag_name": tag_name,
    "name": rel_title,
    "body": f"📺 **Sony Bravia KLV-24P413D USB Ready Media**\n- File: `{final_output}`\n- Size: `{out_mb:.2f} MB`\n- Codec: H.264 Main 3.1 (yuv420p) / AAC Stereo 192k 44.1kHz\n- Native Resolution Invariant: Preserved, zero fake upscaling.",
    "draft": False,
    "prerelease": False
}

resp = requests.post(f"https://api.github.com/repos/{REPO}/releases", headers=headers, json=payload)
if resp.status_code not in [200, 201]:
    raise RuntimeError(f"GitHub release creation failed: {resp.status_code} - {resp.text}")

upload_url = resp.json().get("upload_url", "").split("{")[0]

print(f"[*] Uploading single file {final_output} ({out_mb:.2f} MB) to GitHub Releases...")
up_headers = {
    "Authorization": f"Bearer {gh_token}",
    "Content-Type": "video/mp4"
}

with open(final_output, "rb") as f:
    up_resp = requests.post(f"{upload_url}?name={final_output}", headers=up_headers, data=f, timeout=600)

if up_resp.status_code in [200, 201]:
    asset = up_resp.json()
    download_url = asset.get("browser_download_url")
    print("\n========================================================")
    print("🎉 MOVIE RELEASED SUCCESSFULLY!")
    print(f"📺 File: {final_output}")
    print(f"⬇️ Direct Download Link for your TV:\n{download_url}")
    print("========================================================")
else:
    print(f"[!] Release upload error: {up_resp.status_code} - {up_resp.text}")
